In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer

In [13]:
df = pd.read_csv('./data/preprocessed-data/preprocessed_data.csv')

In [14]:
# Feature Creation
df['Is_Alone'] = ((df['Family_Size'] == 1) | (df['Ever_Married'] == 'No')).astype(int)
df['Career_Stability'] = df['Work_Experience'] / df['Age']

In [15]:
# Feature Transformation
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 25, 40, 60, 100], labels=['Young', 'Adult', 'Senior', 'Elder']) # Có thể được Bining lại nếu kết quả phân chia chưa ổn
df['Work_Experience_Log'] = np.log1p(df['Work_Experience'])

In [5]:
df.head(10)

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Is_Alone,Career_Stability,Age_Group,Work_Experience_Log
0,Male,No,22,No,Healthcare,1.0,Low,4.0,1,0.045455,Young,0.693147
1,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,1,0.014925,Elder,0.693147
2,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,0,0.000000,Elder,0.000000
3,Male,Yes,56,No,Artist,0.0,Average,2.0,0,0.000000,Senior,0.000000
4,Male,No,32,Yes,Healthcare,1.0,Low,3.0,1,0.031250,Adult,0.693147
5,Female,No,33,Yes,Healthcare,1.0,Low,3.0,1,0.030303,Adult,0.693147
6,Female,Yes,61,Yes,Engineer,0.0,Low,3.0,0,0.000000,Elder,0.000000
7,Female,Yes,55,Yes,Artist,1.0,Average,4.0,0,0.018182,Senior,0.693147
8,Female,No,26,Yes,Engineer,1.0,Low,3.0,1,0.038462,Adult,0.693147
9,Male,No,19,No,Healthcare,4.0,Low,4.0,1,0.210526,Young,1.609438


In [6]:
import pandas as pd
import numpy as np
from typing import List, Dict, Union, Any

In [ ]:
class CustomStandardScaler:
    def __int__(self):
        self.mean: np.ndarray = None
        self.std: np.ndarray = None
    def fit(self, X: pd.DataFrame) -> 'CustomStandardScaler':
        """Tính toán mean và độ lệch chuẩn (std) để phục vụ cho việc chuẩn hóa
        
        workflow:
        1. Ép type của dataframe đầu vào thành numpy array
        2. Tính mean theo từng cột
        3. Tính std theo từng cột
        4. Check và replace các giá trị std = 0 thành 1 để tránh lỗi chia cho 0
        
        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng chứa các biến liên tục cần chuẩn hóa

        Returns:
            CustomStandardScaler: Trả về chính đối tượng hiện tại
        """
        X = X.values
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0.0] = 1.0
        
        return self
    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """Thực hiện chuẩn hóa dữ liệu dựa trên mean và std đã tính
        workflow:
        1. Ép type của dataframe đầu vào thành numpy array
        2. Trừ đi giá trị mean đã lưu ở hàm fit
        3. Chia cho giá trị std đã lưu ở hàm fit
        4. Trả về ma trận kết quả
        
        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng cần được chuẩn hóa
            
        Returns:
            np.ndarray: Ma trận dữ liệu đã được chuẩn hóa (Z-score)
        """
        X = X.values
        return (X - self.mean)/self.std
    
    def fit_transform(self, X: pd.DataFrame) -> np.ndarray:
        """Thực hiện fit và transform đồng thời trên tập dữ liệu

        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng cần chuẩn hóa
            
        Returns:
            np.ndarray: Ma trận dữ liệu đã được chuẩn hóa
        """
        return self.fit(X).transform(X)

In [8]:
class CustomOrdinalEncoder:
    def __init__(self, categories: List[List[str]]):
        self.categories = categories
        self.mapping : Dict[str, Dict[str, int]] = {}
    def fit(self, X: pd.DataFrame) -> 'CustomOrdinalEncoder':
        """Xây dựng từ điển mapping từ danh mục sang số nguyên
        workflow:
        1. Lấy danh sách tên các cột từ DataFrame
        2. Duyệt qua từng cột và danh sách thứ tự (categories) tương ứng
            2.1. Tạo một dictionary ánh xạ (vd: {'Low': 0, 'Average': 1, 'High': 2}) cho cột đó
            2.2. Lưu trữ mapping vào thuộc tính mapping của class
        
        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng chứa các biến phân loại có thứ bậc
            
        Returns:
            CustomOrdinalEncoder: Trả về chính đối tượng hiện tại (self)
        """
        columns = X.columns
        for idx, col in enumerate(columns):
            # Tạo dictionary ánh xạ giá trị text sang index (0, 1, 2...)
            col_mapping = {val: i for i, val in enumerate(self.categories[idx])}
            self.mapping[col] = col_mapping
            
        return self
    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """Áp dụng mapping để chuyển đổi text thành số nguyên
        workflow:
        1. Tạo một bản sao của DataFrame đầu vào để tránh ghi đè dữ liệu gốc
        2. Duyệt qua từng cột và sử dụng dictionary đã lưu để thay thế giá trị
        3. Chuyển đổi DataFrame kết quả sang định dạng numpy array
        
        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng chứa các biến cần mã hóa
            
        Returns:
            np.ndarray: Ma trận số nguyên đại diện cho các cấp bậc
        """
        X_encoded = X.copy()
        for col, col_mapping in self.mapping.items():
            X_encoded[col] = X_encoded[col].map(col_mapping)
            
        return X_encoded.values
    def fit_transform(self, X: pd.DataFrame) -> np.ndarray:
        """Thực hiện fit và transform đồng thời trên tập dữ liệu

        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng cần mã hóa
            
        Returns:
            np.ndarray: Ma trận dữ liệu đã mã hóa thứ bậc
        """
        return self.fit(X).transform(X)

In [9]:
class CustomOneHotEncoder:
    def __init__(self, drop_first: bool = True):
        self.drop_first = drop_first
        self.categories: Dict[str, List[str]] = {}
        self.feature_names_out: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'CustomOneHotEncoder':
        """Xác định các categories duy nhất cho từng cột và tạo tên đặc trưng mới
        workflow:
        1. Duyệt qua từng cột trong DataFrame đầu vào
            1.1. Lấy danh sách các giá trị duy nhất (unique) và sắp xếp chúng
            1.2. Nếu drop_first=True, loại bỏ giá trị đầu tiên trong danh sách
            1.3. Lưu các giá trị còn lại vào thuộc tính categories
            1.4. Tạo tên cột mới (vd: Gender_Male) và lưu vào feature_names_out
        
        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng chứa các biến định danh
            
        Returns:
            CustomOneHotEncoder: Trả về chính đối tượng hiện tại (self)
        """
        self.feature_names_out = []
        
        for col in X.columns:
            # Lấy các giá trị duy nhất (bỏ qua NaN nếu có) và sắp xếp
            uniques = sorted([x for x in X[col].unique() if pd.notna(x)])
            
            # Xử lý drop='first' để tránh dummy variable trap
            if self.drop_first and len(uniques) > 1:
                uniques = uniques[1:]
                
            self.categories[col] = uniques
            
            # Tạo tên cột mới (VD: Gender_Male)
            for val in uniques:
                self.feature_names_out.append(f"{col}_{val}")
                
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """Chuyển đổi dữ liệu thành các cột nhị phân (0 và 1)
        workflow:
        1. Khởi tạo một ma trận numpy toàn số 0 với kích thước (số hàng, tổng số cột mới)
        2. Duyệt qua từng cột gốc và từng giá trị hạng mục đã lưu
            2.1. Gán giá trị 1 vào ma trận tại các vị trí mà dữ liệu khớp với hạng mục
            2.2. Trả về ma trận nhị phân
        
        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng cần mã hóa
            
        Returns:
            np.ndarray: Ma trận nhị phân
        """
        num_rows = len(X)
        num_cols = len(self.feature_names_out)
        encoded_arr = np.zeros((num_rows, num_cols), dtype=int)
        
        col_idx = 0
        for col in X.columns:
            uniques = self.categories[col]
            for val in uniques:
                # Kiểm tra điều kiện và chuyển boolean thành int (0 hoặc 1)
                encoded_arr[:, col_idx] = (X[col] == val).astype(int)
                col_idx += 1
                
        return encoded_arr

    def fit_transform(self, X: pd.DataFrame) -> np.ndarray:
        """Thực hiện fit và transform đồng thời trên tập dữ liệu

        Args:
            X (pd.DataFrame): Dữ liệu dạng bảng cần mã hóa
            
        Returns:
            np.ndarray: Ma trận nhị phân sau khi mã hóa
        """
        return self.fit(X).transform(X)

In [10]:
scaler = CustomStandardScaler()
ohe = CustomOneHotEncoder(drop_first=True)
orde = CustomOrdinalEncoder(categories=[['Low', 'Average', 'High']])

In [11]:
# Nhóm Numerical
num_cols = ['Age', 'Work_Experience_Log', 'Family_Size', 'Career_Stability']
X_num = scaler.fit_transform(df[num_cols])

# Nhóm Categorical không thứ bậc
cat_cols = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Age_Group']
X_cat = ohe.fit_transform(df[cat_cols])

# Cột Ordinal có thứ bậc
ord_cols = ['Spending_Score']
X_ord = orde.fit_transform(df[ord_cols])

In [12]:
X_processed = np.hstack((X_num, X_cat, X_ord))
X_processed.shape

(7239, 20)

In [16]:
# Encoding
categorical_cols = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Age_Group']
ordinal_col = ['Spending_Score']

# Định nghĩa thứ tự cho Spending_Score (Low < Average < High)
spending_order = [['Low', 'Average', 'High']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Work_Experience_Log', 'Family_Size', 'Career_Stability']),
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
        ('ord', OrdinalEncoder(categories=spending_order), ordinal_col)
    ]
)

In [17]:
X_processed = preprocessor.fit_transform(df)

In [18]:
X_processed.shape

(7239, 20)

In [19]:
import numpy as np
from typing import Union, Optional

class CustomPCA:
    def __init__(self, n_components: Union[int, float, None] = None):
        self.n_components = n_components
        self.eigenvectors: np.ndarray = None
        self.explained_variance_ratio_: np.ndarray = None
        self.mean_: np.ndarray = None
        self.num_dimension: int = 0

    def fit(self, X: np.ndarray) -> 'CustomPCA':
        """Xây dựng mô hình PCA bằng cách tìm các vector riêng và tính tỷ lệ phương sai
        workflow:
        1. Tính mean của từng feature và đưa dữ liệu về tâm
        2. Tính Covariance Matrix của dữ liệu đã centered
        3. Phân rã ma trận hiệp phương sai để lấy các eigenvalues và eigenvectors
        4. Sắp xếp các eigenvalues và eigenvectors theo thứ tự giảm dần (từ quan trọng nhất đến ít quan trọng nhất)
        5. Xác định số lượng thành phần chính (Số lượng chiều thực tế) cần giữ lại dựa trên tham số n_components 
        (là số nguyên hoặc tỷ lệ % như 0.95)
        6. Lưu trữ các eigenvector được chọn vào self.eigenvectors và tỷ lệ phương sai vào self.explained_variance_ratio_
        
        Args:
            X (np.ndarray): Ma trận dữ liệu đầu vào có kích thước (n_samples, n_features)
            
        Returns:
            CustomPCA: Trả về chính đối tượng hiện tại (self)
        """
        
        # Đưa dữ liệu về tâm
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_

        # Tính Covariance Matrix
        cov_matrix = np.cov(X_centered, rowvar=False) # Cấu hình rowvar=False vì các cột là features, các hàng là samples

        # Phân rã Eigen
        # Dùng eigh tốt hơn eig vì ma trận hiệp phương sai luôn đối xứng
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # Sắp xếp giảm dần
        sorted_idx = np.argsort(eigenvalues)[::-1]
        sorted_eigenvalues = eigenvalues[sorted_idx]
        sorted_eigenvectors = eigenvectors[:, sorted_idx]

        # Tính tỷ lệ phương sai
        total_variance = np.sum(sorted_eigenvalues)
        explained_variance_ratio = sorted_eigenvalues / total_variance

        if self.n_components is None:
            self.num_dimension = X.shape[1]
        elif isinstance(self.n_components, float) and 0.0 < self.n_components < 1.0:
            # Tính cumulative variance và tìm điểm cắt (VD: 95%)
            cumulative_variance = np.cumsum(explained_variance_ratio)
            self.num_dimension = np.searchsorted(cumulative_variance, self.n_components) + 1
        elif isinstance(self.n_components, int):
            self.num_dimension = self.n_components
        else:
            raise ValueError("n_components phải là số nguyên, số thực (0-1) hoặc None")

        # components_ trong scikit-learn có shape (n_components, n_features) nên ta cần transpose
        self.eigenvectors = sorted_eigenvectors[:, :self.num_dimension].T
        self.explained_variance_ratio_ = explained_variance_ratio[:self.num_dimension]

        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Project dữ liệu gốc xuống không gian mới với số chiều nhỏ hơn
        workflow:
        1. Đưa dữ liệu đầu vào về tâm bằng cách trừ đi giá trị mean_ đã học ở hàm fit
        2. Nhân ma trận dữ liệu đã centered với ma trận các vector riêng (components_)
        3. Trả về ma trận dữ liệu đã được giảm chiều
        
        Args:
            X (np.ndarray): Ma trận dữ liệu cần giảm chiều
            
        Returns:
            np.ndarray: Ma trận dữ liệu sau khi thực hiện PCA (n_samples, n_components_)
        """
        # Centering dữ liệu dùng mean của tập huấn luyện
        X_centered = X - self.mean_
        
        # Chiếu dữ liệu: X * W (Trong đó W là ma trận vector riêng chuyển vị)
        return np.dot(X_centered, self.eigenvectors.T)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """Thực hiện fit và transform đồng thời trên tập dữ liệu

        Args:
            X (np.ndarray): Ma trận dữ liệu đầu vào
            
        Returns:
            np.ndarray: Ma trận sau khi đã giảm chiều
        """
        return self.fit(X).transform(X)

In [20]:
custom_pca = CustomPCA(n_components=0.95)

In [21]:
X_final_custom = custom_pca.fit_transform(X_processed)

In [22]:
X_final_custom.shape

(7239, 12)

In [23]:
print(f"{np.sum(custom_pca.explained_variance_ratio_):.2%}")

95.66%


In [ ]:
# Feature Extraction (PCA - Giảm chiều) bằng scikit-learn
pca = PCA(n_components=0.95) # Giữ lại 95% phương sai thông tin
X_final = pca.fit_transform(X_processed)

In [25]:
X_final.shape

(7239, 12)

In [26]:
print(f"{sum(pca.explained_variance_ratio_):.2%}")

95.66%


In [27]:
feature_names = preprocessor.get_feature_names_out()

# Trích xuất loadings vào một DataFrame
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=[f'PC{i+1}' for i in range(pca.n_components_)], 
    index=feature_names
)

# Hiển thị 5 thành phần chính đầu tiên
print(loadings.head(10))

                                    PC1       PC2       PC3       PC4  \
num__Age                      -0.452312 -0.530177  0.269012 -0.497981   
num__Work_Experience_Log       0.558974 -0.419373  0.193668 -0.053025   
num__Family_Size               0.105887  0.558456  0.744461 -0.267050   
num__Career_Stability          0.617385 -0.283976  0.125367 -0.061066   
cat__Gender_Male              -0.022766  0.015237  0.055259  0.075586   
cat__Ever_Married_Yes         -0.145261 -0.178414  0.224076  0.239756   
cat__Graduated_Yes            -0.050362 -0.137399 -0.019900  0.160221   
cat__Profession_Doctor         0.010088  0.018637 -0.024413  0.019911   
cat__Profession_Engineer       0.001739  0.007280 -0.000824  0.000953   
cat__Profession_Entertainment  0.002017 -0.002755 -0.011778  0.011017   

                                    PC5       PC6       PC7       PC8  \
num__Age                       0.107568  0.050791 -0.087156  0.141042   
num__Work_Experience_Log       0.134078  0.052439 

In [ ]:
import pickle

df_pca = pd.DataFrame(X_final_custom, columns=[f'PC{i+1}' for i in range(X_final_custom.shape[1])])
# Tập Train
df_pca.to_csv('./data/ready_for_train/X_features.csv', index=False)

In [ ]:
# Dùng để giải thích mô hình 
df.to_csv('./data/ready_for_train/cleaned_customer_data.csv', index=False)

In [ ]:
custom_data_encoding = {
    'scaler': scaler,
    'one_hot': ohe,
    'ordinal': orde,
    'pca': custom_pca
}

# Dùng cho inference
with open('./data/ready_for_train/custom_data_encoding.pkl', 'wb') as f:
    pickle.dump(custom_data_encoding, f)